In [1]:
import os
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from tensorflow.keras.models import load_model
import threading
import pygame

# Initialize pygame for alarm sound
pygame.mixer.init()
ALARM_SOUND = "alarm.wav"
alarm_sound = pygame.mixer.Sound(ALARM_SOUND)

# Mediapipe initialization
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Load knife detection model
weapon_model = load_model("knife_classifier_model.h5")
classes = ["not_knife", "knife"]

# Directory to save detected frames
SAVE_DIR = "detected_images"
os.makedirs(SAVE_DIR, exist_ok=True)

# Global variables for alarm
alarm_playing = False
alarm_lock = threading.Lock()

def preprocess_frame(frame, img_height=224, img_width=224):
    """Preprocess the frame for knife classification."""
    image = cv2.resize(frame, (img_height, img_width), interpolation=cv2.INTER_AREA)
    image = image / 255.0
    return np.expand_dims(image, axis=0)

def draw_text_with_background(frame, text, position, font=cv2.FONT_HERSHEY_SIMPLEX, font_scale=0.5, 
                              text_color=(255, 255, 255), background_color=(0, 0, 255), thickness=1):
    """Draw text with a background rectangle."""
    text_size, _ = cv2.getTextSize(text, font, font_scale, thickness)
    text_w, text_h = text_size
    x, y = position
    cv2.rectangle(frame, (x, y - text_h - 5), (x + text_w + 10, y + 5), background_color, -1)
    cv2.putText(frame, text, (x, y), font, font_scale, text_color, thickness)

def save_detected_frame(frame, label):
    """Save the current frame with a unique filename."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{SAVE_DIR}/{label}_{timestamp}.jpg"
    cv2.imwrite(filename, frame)
    print(f"Saved image: {filename}")

def play_alarm():
    """Play the alarm sound."""
    global alarm_playing
    with alarm_lock:
        if not alarm_playing:
            alarm_playing = True
            alarm_sound.play()
            time.sleep(alarm_sound.get_length())  # Wait for the alarm to finish
            alarm_playing = False

cap = cv2.VideoCapture(r"C:\Users\Andhavarapu Jahnavi\Desktop\IntelliGuard Multi-Modal AI Threat Detection System\sample2.mp4")

# Detection thresholds
knife_confidence_threshold = 0.85
frame_skip = 2
frame_counter = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    frame_counter += 1
    if frame_counter % frame_skip != 0:  # Skip frames for efficiency
        continue

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pose_results = pose.process(image_rgb)

    # Unusual behavior detection
    if pose_results.pose_landmarks:
        left_hand_y = pose_results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_WRIST].y
        right_hand_y = pose_results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].y
        nose_y = pose_results.pose_landmarks.landmark[mp_pose.PoseLandmark.NOSE].y

        if left_hand_y < nose_y or right_hand_y < nose_y:
            frame_height, frame_width, _ = frame.shape
            draw_text_with_background(frame, "Unusual Behavior Detected!", 
                                      position=(10, frame_height - 30), font_scale=0.7, 
                                      background_color=(255, 0, 0))
            save_detected_frame(frame, "unusual_behavior")

    # Weapon detection
    preprocessed_frame = preprocess_frame(frame)
    prediction = weapon_model.predict(preprocessed_frame, verbose=0)[0][0]
    predicted_class = classes[1] if prediction > 0.5 else classes[0]
    confidence = prediction if prediction > 0.5 else 1 - prediction

    if predicted_class == "knife" and confidence >= knife_confidence_threshold:
        threading.Thread(target=play_alarm, daemon=True).start()
        draw_text_with_background(frame, f"Weapon Detected: {predicted_class}", (10, 30), font_scale=0.7)
        draw_text_with_background(frame, f"Confidence: {confidence:.2f}", (10, 60), font_scale=0.7)
        save_detected_frame(frame, "weapon_detected")

    cv2.imshow('Surveillance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

pygame 2.6.1 (SDL 2.28.4, Python 3.10.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


Saved image: detected_images/weapon_detected_2026-02-05_22-58-30.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-30.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-31.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-32.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-32.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-32.jpg
Saved image: detected_images/weapon_detected_2026-02-05_22-58-32.jpg
Saved image: detected_images/weapo